In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd

news_path = 'data/data/news.csv'
news_full_path = 'data/data/news_full.csv'

news = pd.read_csv(news_path)
news_full = pd.read_csv(news_full_path)


In [ ]:
issuer_data = pd.read_csv('data/new_data/issuer_data_textual.csv')

In [ ]:
import pandas as pd

# --------------------- Load raw news data ---------------------
# Read the full news dataset containing headlines and article text
news_full_path = 'data/data/news_full.csv'
news = pd.read_csv(news_full_path)
news["dt"] = pd.to_datetime(news["dt"])

# --------------------- Text field preprocessing ---------------------
# Fill missing values in textual fields to avoid issues during concatenation
news["headline"] = news["headline"].fillna("")
news["plain_txt"] = news["plain_txt"].fillna("")

# --------------------- Aggregate news at the equity-day level ---------------------
# Group news articles by equity identifier and date,
# and concatenate all headlines and article texts within the same day
news_daily = (
    news.groupby(["fs_entity_id", "dt"])
    .agg({
        "headline": lambda x: " ".join(x),
        "plain_txt": lambda x: " ".join(x)
    })
    .reset_index()
)

# --------------------- Construct final merged text ---------------------
# Combine daily aggregated headlines and article bodies into a single text field
news_daily["merged_text"] = news_daily["headline"] + " " + news_daily["plain_txt"]
news_daily = news_daily[["fs_entity_id", "dt", "merged_text"]]


In [ ]:
issuer_events = issuer_data[issuer_data['event_count'] > 0].copy()

In [ ]:
issuer_events["dt"] = pd.to_datetime(issuer_events["dt"])
news_daily["dt"] = pd.to_datetime(news_daily["dt"])


In [ ]:
issuer_events = issuer_events.merge(
    news_daily,
    on=["fs_entity_id", "dt"],
    how="left"
)


In [ ]:
import pandas as pd

# --------------------- Prepare news data for efficient lookup ---------------------
# Create a working copy and ensure the date column is in datetime format
news_daily = news_daily.copy()
news_daily["dt"] = pd.to_datetime(news_daily["dt"])

def build_final_text(row, news_daily, lookback_days=3):
    """
    Construct the final text used for sentiment analysis.

    For each equity-day observation:
    - If news exists on the same day, use the same-day merged text.
    - Otherwise, aggregate news from the previous `lookback_days` calendar days
      (excluding the current day) as a fallback.
    """
    if isinstance(row["merged_text"], str) and row["merged_text"].strip() != "":
        return row["merged_text"]

    eid = row["fs_entity_id"]
    d = row["dt"]
    start = d - pd.Timedelta(days=lookback_days)

    mask = (
        (news_daily["fs_entity_id"] == eid) &
        (news_daily["dt"] >= start) &
        (news_daily["dt"] < d)
    )
    texts = news_daily.loc[mask, "merged_text"].dropna().tolist()

    if len(texts) == 0:
        return ""
    else:
        return " ".join(texts)

# --------------------- Construct final text field for sentiment modeling ---------------------
# Apply the text construction logic row by row:
# - Same-day news if available
# - Otherwise, aggregated news from the previous 3 days
issuer_events["final_text"] = issuer_events.apply(
    build_final_text,
    axis=1,
    news_daily=news_daily,
    lookback_days=3
)


In [ ]:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# import torch
# import numpy as np

# model_name = "yiyanghkust/finbert-tone"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# model.eval()

# def compute_sent(text):
#     if not isinstance(text, str) or text.strip() == "":
#         return 0.0
#     inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
#     with torch.no_grad():
#         logits = model(**inputs).logits
#         probs = torch.softmax(logits, dim=1)[0].tolist()
#     neg, neu, pos = probs
#     return pos - neg


In [ ]:
issuer_events.final_text.str.len().describe()

In [ ]:
# --------------------- Remove observations with empty text ---------------------
# Fill missing text values to avoid NaN issues when computing string length
issuer_events["final_text"] = issuer_events["final_text"].fillna("")
issuer_events["final_len"] = issuer_events["final_text"].str.len()

# Compute text length for each observation
issuer_events_cleaned = issuer_events[issuer_events["final_len"] > 0].copy()

print("Original number of rows:", len(issuer_events))
print("Number of rows after removing empty text:", len(issuer_events_cleaned))
print("Number of rows removed:", len(issuer_events) - len(issuer_events_cleaned))


In [ ]:
issuer_events_cleaned.to_csv(
    "data/new_data/issuer_events_cleaned_textual_data.csv",
    index=False
)
